# Lab 8: Complete RAG Pipeline
## Retrieval Comparison, Context Building, and Prompt Writing

This lab covers the full pipeline from raw documents to a prompt-ready evidence package.

The lab is organized into three connected parts:

| Part | Topic |
|---|---|
| Part 1 | Retrieval Comparison — TF-IDF vs Embeddings vs Hybrid |
| Part 2 | Context Building — From retrieved chunks to usable evidence |
| Part 3 | Prompt Writing — How to write prompts that use evidence well |

The central question throughout is:

> How do we go from a user query and a knowledge base to a grounded, trustworthy answer?

## The Big Picture

```text
documents
→ chunks
→ retriever (TF-IDF / Embeddings / Hybrid)
→ candidate evidence
→ context building (filter, order, deduplicate)
→ context package
→ prompt (weak / better / strict)
→ LLM answer
```

Each step can be the source of failure. This lab teaches you to control each step.

## Learning Outcomes

By the end of this lab, you should be able to explain and implement:

| Topic | What you must understand |
|---|---|
| TF-IDF retrieval | Lexical baseline for exact words and phrases |
| Embedding retrieval | Semantic similarity using dense vectors |
| Hybrid retrieval | Why combining lexical and semantic signals is often strongest |
| Retrieval metrics | Precision@K, Recall@K, Hit Rate@K, MRR |
| Context building | Filter, order, deduplicate, and budget control |
| Metadata handling | Current vs outdated sources, conflict detection |
| Prompt anatomy | Role, task, evidence boundary, rules, output format |
| Prompt styles | Weak, better, and strict prompts and when each fits |
| Failure analysis | How to separate retrieval, context, and prompt failures |

# Section 1 — Install Required Packages

Run this cell once if the packages are not already installed.

In [1]:
!pip install -q sentence-transformers rank-bm25 scikit-learn


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\ahmed\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


# Section 2 — Import Libraries

In [2]:
import re
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

pd.set_option("display.max_colwidth", 220)
pd.set_option("display.max_columns", 160)

# Section 3 — Create the Knowledge Base

We use a realistic student-services corpus.

The corpus is deliberately messy:

- overlapping topics across documents
- exact numbers, dates, and codes that matter
- paraphrases and synonyms that test semantic retrieval
- role-specific documents (student vs staff)
- two outdated documents that conflict with current policy

Each document carries metadata that context building will use later.

In [3]:
documents = [
    {
        "document_id": 0,
        "title": "Library Borrowing Policy",
        "department": "Library",
        "doc_type": "policy",
        "effective_date": "2025-08-01",
        "is_current": True,
        "text": (
            "Students may borrow up to 8 printed books at one time for 14 days. "
            "Renewals are allowed twice unless another student has placed a hold. "
            "Graduating students must clear all outstanding loans and fines before clearance is approved. "
            "Reference materials, thesis copies, and special reserve items cannot be borrowed overnight."
        )
    },
    {
        "document_id": 1,
        "title": "Library Research Database Access",
        "department": "Library",
        "doc_type": "help page",
        "effective_date": "2025-09-10",
        "is_current": True,
        "text": (
            "Students can access research databases on campus automatically through the university network. "
            "Off-campus access requires signing in with the university email, password, and multi-factor verification. "
            "If a database page loads but articles remain locked, students should reconnect through the library portal rather than search the publisher site directly."
        )
    },
    {
        "document_id": 2,
        "title": "Password Reset for Students",
        "department": "IT Services",
        "doc_type": "procedure",
        "effective_date": "2025-09-01",
        "is_current": True,
        "text": (
            "Students must use the self-service password portal to reset a forgotten account password. "
            "The portal asks for student ID, university email, and a verification code sent to the registered mobile number. "
            "If the mobile number is outdated, the student must visit the IT service desk with a valid ID card before access is restored."
        )
    },
    {
        "document_id": 3,
        "title": "Password Reset for Staff",
        "department": "IT Services",
        "doc_type": "procedure",
        "effective_date": "2025-09-01",
        "is_current": True,
        "text": (
            "Staff members must reset passwords through the employee identity portal. "
            "A payroll number is required for verification. "
            "Academic staff who cannot access the employee portal should contact departmental IT support. "
            "Student IDs cannot be used in this workflow."
        )
    },
    {
        "document_id": 4,
        "title": "Tuition Refund Policy",
        "department": "Finance",
        "doc_type": "policy",
        "effective_date": "2025-08-15",
        "is_current": True,
        "text": (
            "Students who withdraw from all courses during the first 7 calendar days of the semester may receive an 80 percent tuition refund. "
            "Withdrawals during days 8 to 14 may receive a 50 percent refund. "
            "Registration fees, late-payment penalties, and technology fees are non-refundable. "
            "Refund processing normally takes 10 working days after approval."
        )
    },
    {
        "document_id": 5,
        "title": "Course Withdrawal Rules",
        "department": "Registration",
        "doc_type": "policy",
        "effective_date": "2025-08-20",
        "is_current": True,
        "text": (
            "Withdrawing from a course removes the student from academic attendance in that course. "
            "Withdrawal approval does not automatically mean tuition will be refunded. "
            "Refund eligibility is governed by the separate tuition refund policy. "
            "Students who remain enrolled in other courses may still owe full fees depending on timing and load."
        )
    },
    {
        "document_id": 6,
        "title": "Campus Printing Guide",
        "department": "Student Computing",
        "doc_type": "help page",
        "effective_date": "2025-09-05",
        "is_current": True,
        "text": (
            "Students receive a subsidized printing quota of 150 black-and-white pages each semester. "
            "Black-and-white pages after the quota cost 0.10 USD per page, while color printing costs 0.50 USD per page from the first page. "
            "Failed print jobs may be refunded if reported to the lab supervisor within 24 hours with the print job reference number."
        )
    },
    {
        "document_id": 7,
        "title": "Health Clinic Appointment Policy",
        "department": "Health Services",
        "doc_type": "policy",
        "effective_date": "2025-07-20",
        "is_current": True,
        "text": (
            "Routine clinic appointments must be booked in advance through the health portal or reception desk. "
            "Walk-ins are not accepted for routine consultations. "
            "Students who need urgent care for injury, severe pain, or breathing difficulty may be seen without a booking. "
            "Cancellations should be made at least 2 hours before the appointment time."
        )
    },
    {
        "document_id": 8,
        "title": "Internship Registration Requirements",
        "department": "Career Services",
        "doc_type": "procedure",
        "effective_date": "2025-09-12",
        "is_current": True,
        "text": (
            "Students registering for internship credit must have completed at least 60 credit hours and hold a cumulative GPA of 2.5 or above. "
            "Required documents include the internship offer letter, learning plan, and department approval form. "
            "All internship registrations for the fall term must be submitted by October 10."
        )
    },
    {
        "document_id": 9,
        "title": "Academic Appeals Procedure",
        "department": "Academic Affairs",
        "doc_type": "procedure",
        "effective_date": "2025-09-18",
        "is_current": True,
        "text": (
            "Students who wish to appeal an academic decision must submit form AP-17 with supporting evidence. "
            "The appeal should explain the decision being challenged, the reason for the request, and any relevant medical or administrative documents. "
            "Appeals must be submitted within 10 working days of the official decision notice. "
            "Late appeals are normally rejected unless the student documents exceptional circumstances. "
            "After submission, the faculty committee reviews the case and may request an interview or additional evidence."
        )
    },
    {
        "document_id": 10,
        "title": "Financial Aid Disbursement Notice",
        "department": "Financial Aid",
        "doc_type": "notice",
        "effective_date": "2025-09-03",
        "is_current": True,
        "text": (
            "Approved financial aid is usually disbursed after enrollment verification is complete. "
            "Students may see a delay if required documents are missing, if registration changes after the census date, or if the bank account details are invalid. "
            "Receiving financial aid does not cancel library fines, housing charges, or printing balances."
        )
    },
    {
        "document_id": 11,
        "title": "Parking Permit Rules",
        "department": "Campus Operations",
        "doc_type": "policy",
        "effective_date": "2025-08-22",
        "is_current": True,
        "text": (
            "Semester parking permits cost 120 USD for students and are linked to one registered vehicle. "
            "Permit requests require the student ID, vehicle registration, and a valid campus account. "
            "Parking in staff-only zones without authorization may lead to fines or permit suspension."
        )
    },
    {
        "document_id": 12,
        "title": "Student ID Card Replacement",
        "department": "Student Affairs",
        "doc_type": "procedure",
        "effective_date": "2025-08-28",
        "is_current": True,
        "text": (
            "Replacing a lost student ID card costs 20 USD. "
            "Students must present a government ID or passport when requesting a replacement. "
            "The temporary access slip allows entry to the library for one day but does not work for printing or attendance scanners."
        )
    },
    {
        "document_id": 13,
        "title": "Graduation Clearance Checklist",
        "department": "Registrar",
        "doc_type": "checklist",
        "effective_date": "2025-09-14",
        "is_current": True,
        "text": (
            "Graduation clearance requires confirmation from the library, finance office, housing office, and academic department. "
            "Unpaid fines, overdue books, missing equipment, or unresolved tuition balances can block clearance. "
            "Transcript release may be delayed until all holds are removed."
        )
    },
    {
        "document_id": 14,
        "title": "Housing Move-Out Instructions",
        "department": "Residence Services",
        "doc_type": "instruction",
        "effective_date": "2025-05-30",
        "is_current": True,
        "text": (
            "Residents must complete move-out inspection before returning room keys. "
            "Late checkout may result in an extra nightly charge. "
            "Lost keys incur a replacement fee, and excessive room damage is billed after final inspection. "
            "Students should clear all personal items before departure."
        )
    },
    {
        "document_id": 15,
        "title": "IT Service Desk FAQ",
        "department": "IT Services",
        "doc_type": "FAQ",
        "effective_date": "2025-09-16",
        "is_current": True,
        "text": (
            "The IT service desk handles password issues, campus Wi-Fi support, account lockouts, software access, and printer login problems. "
            "Some issues can be resolved remotely, but identity-sensitive account recovery may require an in-person visit. "
            "Students and staff should use the correct portal for their role before requesting manual support."
        )
    },
    {
        "document_id": 16,
        "title": "Refund Notice for Spring 2024",
        "department": "Finance",
        "doc_type": "old notice",
        "effective_date": "2024-01-10",
        "is_current": False,
        "text": (
            "Effective for Spring 2024 only, students who withdraw during the first 10 calendar days may receive a 75 percent tuition refund. "
            "This temporary schedule replaced the normal refund table for that term only."
        )
    },
    {
        "document_id": 17,
        "title": "Old Printing Price Notice",
        "department": "Student Computing",
        "doc_type": "old notice",
        "effective_date": "2023-11-01",
        "is_current": False,
        "text": (
            "Before the 2025 pricing update, color printing cost 0.35 USD per page and black-and-white printing cost 0.08 USD per page after quota. "
            "This notice is retained for record purposes and should not be used for current billing questions."
        )
    }
]

documents_df = pd.DataFrame(documents)
documents_df[["document_id", "title", "department", "doc_type", "effective_date", "is_current"]]

,document_id,title,department,doc_type,effective_date,is_current
0,0,Library Borrowing Policy,Library,policy,2025-08-01,True
1,1,Library Research Database Access,Library,help page,2025-09-10,True
2,2,Password Reset for Students,IT Services,procedure,2025-09-01,True
3,3,Password Reset for Staff,IT Services,procedure,2025-09-01,True
4,4,Tuition Refund Policy,Finance,policy,2025-08-15,True
5,5,Course Withdrawal Rules,Registration,policy,2025-08-20,True
6,6,Campus Printing Guide,Student Computing,help page,2025-09-05,True
7,7,Health Clinic Appointment Policy,Health Services,policy,2025-07-20,True
8,8,Internship Registration Requirements,Career Services,procedure,2025-09-12,True
9,9,Academic Appeals Procedure,Academic Affairs,procedure,2025-09-18,True


## Why This Corpus Creates Real Retrieval Pressure

| Pressure Type | Example |
|---|---|
| Paraphrase | `money back` vs `refund` |
| Role-specific | student password reset vs staff password reset |
| Exact detail | prices, dates, percentages, codes |
| Outdated vs current | Spring 2024 refund notice vs current refund policy |
| Multi-document | graduation clearance depends on library + finance + housing |

# Section 4 — Queries and Ground Truth

Ground truth defines which documents are correct for each query.

Without it, you cannot measure retrieval quality objectively.

In [4]:
ground_truth = {
    "How can I get my money back after dropping classes?": [4, 5],
    "If I withdraw from one course, do I automatically get a refund?": [5, 4],
    "How do I access journal articles from home?": [1],
    "Can I walk into the clinic without booking first?": [7],
    "How much is color printing now?": [6],
    "How much does replacing my student ID cost?": [12],
    "What form code is needed for an academic appeal?": [9],
    "What do I need for an academic appeal and when is it due?": [9],
    "I am staff and cannot access my account. What should I use to reset it?": [3],
    "How do I reset my password?": [2, 15],
    "What can block my graduation clearance?": [13, 0, 10],
    "How many books can I borrow and for how long?": [0],
    "What documents are needed for internship registration?": [8],
}

queries_df = pd.DataFrame({
    "query": list(ground_truth.keys()),
    "relevant_document_ids": list(ground_truth.values())
})
queries_df

,query,relevant_document_ids
0,How can I get my money back after dropping classes?,"[4, 5]"
1,"If I withdraw from one course, do I automatically get a refund?","[5, 4]"
2,How do I access journal articles from home?,[1]
3,Can I walk into the clinic without booking first?,[7]
4,How much is color printing now?,[6]
5,How much does replacing my student ID cost?,[12]
6,What form code is needed for an academic appeal?,[9]
7,What do I need for an academic appeal and when is it due?,[9]
8,I am staff and cannot access my account. What should I use to reset it?,[3]
9,How do I reset my password?,"[2, 15]"


# Section 5 — Chunk the Documents

We split documents into chunks because:

- a long document may contain the answer in only one paragraph
- retrieving the whole document wastes context budget
- chunk-level scoring is more precise than document-level scoring

Each chunk carries full metadata from its source document so we can filter later.

In [5]:
def chunk_text(text, chunk_size=38, overlap=10):
    words = text.split()
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if overlap < 0:
        raise ValueError("overlap cannot be negative")
    if overlap >= chunk_size:
        raise ValueError("overlap must be smaller than chunk_size")

    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunks.append(" ".join(words[start:end]))
        if end >= len(words):
            break
        start += chunk_size - overlap
    return chunks


chunk_rows = []
for doc in documents:
    for chunk_index, chunk_text_value in enumerate(chunk_text(doc["text"])):
        chunk_rows.append({
            "chunk_id": f"doc{doc['document_id']}_chunk{chunk_index}",
            "document_id": doc["document_id"],
            "title": doc["title"],
            "department": doc["department"],
            "doc_type": doc["doc_type"],
            "effective_date": doc["effective_date"],
            "is_current": doc["is_current"],
            "chunk_index": chunk_index,
            "chunk_text": chunk_text_value,
            "search_text": f"{doc['title']} {doc['department']} {doc['doc_type']} {chunk_text_value}"
        })

chunks_df = pd.DataFrame(chunk_rows)
print("Number of source documents:", len(documents_df))
print("Number of chunks:", len(chunks_df))
chunks_df.head(8)

Number of source documents: 18
Number of chunks: 33


,chunk_id,document_id,title,department,doc_type,effective_date,is_current,chunk_index,chunk_text,search_text
0,doc0_chunk0,0,Library Borrowing Policy,Library,policy,2025-08-01,True,0,Students may borrow up to 8 printed books at one time for 14 days. Renewals are allowed twice unless another student has placed a hold. Graduating students must clear all outstanding loans and fines before clearance ...,Library Borrowing Policy Library policy Students may borrow up to 8 printed books at one time for 14 days. Renewals are allowed twice unless another student has placed a hold. Graduating students must clear all outst...
1,doc0_chunk1,0,Library Borrowing Policy,Library,policy,2025-08-01,True,1,"clear all outstanding loans and fines before clearance is approved. Reference materials, thesis copies, and special reserve items cannot be borrowed overnight.","Library Borrowing Policy Library policy clear all outstanding loans and fines before clearance is approved. Reference materials, thesis copies, and special reserve items cannot be borrowed overnight."
2,doc1_chunk0,1,Library Research Database Access,Library,help page,2025-09-10,True,0,"Students can access research databases on campus automatically through the university network. Off-campus access requires signing in with the university email, password, and multi-factor verification. If a database p...","Library Research Database Access Library help page Students can access research databases on campus automatically through the university network. Off-campus access requires signing in with the university email, passw..."
3,doc1_chunk1,1,Library Research Database Access,Library,help page,2025-09-10,True,1,"page loads but articles remain locked, students should reconnect through the library portal rather than search the publisher site directly.","Library Research Database Access Library help page page loads but articles remain locked, students should reconnect through the library portal rather than search the publisher site directly."
4,doc2_chunk0,2,Password Reset for Students,IT Services,procedure,2025-09-01,True,0,"Students must use the self-service password portal to reset a forgotten account password. The portal asks for student ID, university email, and a verification code sent to the registered mobile number. If the mobile ...","Password Reset for Students IT Services procedure Students must use the self-service password portal to reset a forgotten account password. The portal asks for student ID, university email, and a verification code se..."
5,doc2_chunk1,2,Password Reset for Students,IT Services,procedure,2025-09-01,True,1,"registered mobile number. If the mobile number is outdated, the student must visit the IT service desk with a valid ID card before access is restored.","Password Reset for Students IT Services procedure registered mobile number. If the mobile number is outdated, the student must visit the IT service desk with a valid ID card before access is restored."
6,doc3_chunk0,3,Password Reset for Staff,IT Services,procedure,2025-09-01,True,0,Staff members must reset passwords through the employee identity portal. A payroll number is required for verification. Academic staff who cannot access the employee portal should contact departmental IT support. Stu...,Password Reset for Staff IT Services procedure Staff members must reset passwords through the employee identity portal. A payroll number is required for verification. Academic staff who cannot access the employee por...
7,doc4_chunk0,4,Tuition Refund Policy,Finance,policy,2025-08-15,True,0,"Students who withdraw from all courses during the first 7 calendar days of the semester may receive an 80 percent tuition refund. Withdrawals during days 8 to 14 may receive a 50 percent refund. Registration fees, la...",Tuition Refund Policy Finance policy Students who withdraw from all courses during the first 7 calendar days of the semester may receive an 80 percent tuition refund. Withdrawals during days 8 to 14 may recei

## Why `search_text` Includes Metadata

We prepend the title, department, and doc_type to the chunk text.

This helps both TF-IDF and embedding retrieval use the document context, not just the chunk words alone.

Example:
```text
"Tuition Refund Policy Finance policy Students who withdraw..."
```

A query about `refund` will score higher on this chunk because the title reinforces the topic.

# Section 6 — Retrieval Metrics

We use the same metrics for all three retrievers so the comparison is fair.

For chunk retrieval we evaluate at **K = 3**: the top 3 returned chunks.

| Metric | What it measures |
|---|---|
| Precision@K | How many of the top-K results are relevant |
| Recall@K | How many relevant documents appear in the top-K |
| Hit Rate@K | Did at least one relevant document appear in the top-K? (1 or 0) |
| Reciprocal Rank | Inverse of the rank of the first relevant result |

In [6]:
def precision_at_k(retrieved_ids, relevant_ids, k):
    hits = set(retrieved_ids[:k]).intersection(set(relevant_ids))
    return len(hits) / k


def recall_at_k(retrieved_ids, relevant_ids, k):
    hits = set(retrieved_ids[:k]).intersection(set(relevant_ids))
    return len(hits) / len(relevant_ids)


def hit_rate_at_k(retrieved_ids, relevant_ids, k):
    hits = set(retrieved_ids[:k]).intersection(set(relevant_ids))
    return int(len(hits) > 0)


def reciprocal_rank(retrieved_ids, relevant_ids):
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in set(relevant_ids):
            return 1 / rank
    return 0.0


def evaluate_retriever(retriever_name, retrieval_function, ground_truth, k=3):
    rows = []
    for query, relevant_ids in ground_truth.items():
        results = retrieval_function(query, k)
        retrieved_doc_ids = results["document_id"].tolist()
        rows.append({
            "retriever": retriever_name,
            "query": query,
            "relevant_ids": relevant_ids,
            "retrieved_doc_ids": retrieved_doc_ids,
            f"precision@{k}": precision_at_k(retrieved_doc_ids, relevant_ids, k),
            f"recall@{k}": recall_at_k(retrieved_doc_ids, relevant_ids, k),
            f"hit_rate@{k}": hit_rate_at_k(retrieved_doc_ids, relevant_ids, k),
            "reciprocal_rank": reciprocal_rank(retrieved_doc_ids, relevant_ids),
        })
    return pd.DataFrame(rows)

# Section 7 — TF-IDF Retriever

TF-IDF is the lexical baseline.

It scores a chunk highly when the query and chunk share important words.

It fails when the query uses synonyms or paraphrases that are not in the chunk text.

We use bigrams (`ngram_range=(1, 2)`) to catch phrase-level patterns like `student id` or `password reset`.

In [7]:
def normalize_lexical_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s\-]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = tfidf_vectorizer.fit_transform(
    chunks_df["search_text"].map(normalize_lexical_text)
)

print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Rows = chunks, Columns = vocabulary terms")

TF-IDF matrix shape: (33, 1274)
Rows = chunks, Columns = vocabulary terms


In [8]:
def retrieve_top_k_tfidf(query, k=3):
    query_vector = tfidf_vectorizer.transform([normalize_lexical_text(query)])
    scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    ranking = np.argsort(scores)[::-1][:k]

    results = chunks_df.iloc[ranking].copy()
    results["score"] = scores[ranking]
    results["retriever"] = "TF-IDF"
    return results[["retriever", "chunk_id", "document_id", "title",
                    "effective_date", "is_current", "score", "chunk_text"]].reset_index(drop=True)


# Test on a lexical trap query
retrieve_top_k_tfidf("How can I get my money back after dropping classes?", k=5)

,retriever,chunk_id,document_id,title,effective_date,is_current,score,chunk_text
0,TF-IDF,doc10_chunk0,10,Financial Aid Disbursement Notice,2025-09-03,True,0.075501,"Approved financial aid is usually disbursed after enrollment verification is complete. Students may see a delay if required documents are missing, if registration changes after the census date, or if the bank account..."
1,TF-IDF,doc1_chunk0,1,Library Research Database Access,2025-09-10,True,0.072447,"Students can access research databases on campus automatically through the university network. Off-campus access requires signing in with the university email, password, and multi-factor verification. If a database p..."
2,TF-IDF,doc13_chunk0,13,Graduation Clearance Checklist,2025-09-14,True,0.072429,"Graduation clearance requires confirmation from the library, finance office, housing office, and academic department. Unpaid fines, overdue books, missing equipment, or unresolved tuition balances can block clearance..."
3,TF-IDF,doc15_chunk0,15,IT Service Desk FAQ,2025-09-16,True,0.070740,"The IT service desk handles password issues, campus Wi-Fi support, account lockouts, software access, and printer login problems. Some issues can be resolved remotely, but identity-sensitive account recovery may requ..."
4,TF-IDF,doc14_chunk1,14,Housing Move-Out Instructions,2025-05-30,True,0.068575,is billed after final inspection. Students should clear all personal items before departure.


## TF-IDF Failure Analysis

The query uses `money back` and `dropping classes`.

The relevant documents use `refund` and `withdrawal`.

TF-IDF will miss this match because there is no word overlap between the query and the correct documents.

This is the core problem that semantic retrieval solves.

# Section 8 — BM25 Retriever

BM25 is a stronger lexical retriever than TF-IDF.

It handles:
- term frequency saturation (very frequent terms do not dominate)
- document length normalization (longer documents are not unfairly boosted)

But like TF-IDF, BM25 still depends on word overlap. It cannot match synonyms.

In [9]:
def simple_tokenize(text):
    return re.findall(r"[a-z0-9]+", text.lower())


tokenized_chunks = [simple_tokenize(text) for text in chunks_df["search_text"]]
bm25 = BM25Okapi(tokenized_chunks)


def retrieve_top_k_bm25(query, k=3):
    tokenized_query = simple_tokenize(query)
    scores = bm25.get_scores(tokenized_query)
    ranking = np.argsort(scores)[::-1][:k]

    results = chunks_df.iloc[ranking].copy()
    results["score"] = scores[ranking]
    results["retriever"] = "BM25"
    return results[["retriever", "chunk_id", "document_id", "title",
                    "effective_date", "is_current", "score", "chunk_text"]].reset_index(drop=True)


retrieve_top_k_bm25("How can I get my money back after dropping classes?", k=5)

,retriever,chunk_id,document_id,title,effective_date,is_current,score,chunk_text
0,BM25,doc13_chunk0,13,Graduation Clearance Checklist,2025-09-14,True,2.055235,"Graduation clearance requires confirmation from the library, finance office, housing office, and academic department. Unpaid fines, overdue books, missing equipment, or unresolved tuition balances can block clearance..."
1,BM25,doc1_chunk0,1,Library Research Database Access,2025-09-10,True,1.944606,"Students can access research databases on campus automatically through the university network. Off-campus access requires signing in with the university email, password, and multi-factor verification. If a database p..."
2,BM25,doc15_chunk0,15,IT Service Desk FAQ,2025-09-16,True,1.923895,"The IT service desk handles password issues, campus Wi-Fi support, account lockouts, software access, and printer login problems. Some issues can be resolved remotely, but identity-sensitive account recovery may requ..."
3,BM25,doc10_chunk0,10,Financial Aid Disbursement Notice,2025-09-03,True,1.475295,"Approved financial aid is usually disbursed after enrollment verification is complete. Students may see a delay if required documents are missing, if registration changes after the census date, or if the bank account..."
4,BM25,doc14_chunk1,14,Housing Move-Out Instructions,2025-05-30,True,1.391160,is billed after final inspection. Students should clear all personal items before departure.


# Section 9 — Embedding Retriever

An embedding is a dense numerical vector that represents text meaning.

Unlike TF-IDF and BM25 which match words, embeddings match **meaning**.

A sentence embedding model converts any text into a fixed-length vector.
Texts with similar meaning produce vectors that point in similar directions.



## Load the Sentence Embedding Model

We use `all-MiniLM-L6-v2` — a small, fast, and effective model for semantic similarity.

The first run will download the model. Subsequent runs load it from cache.

In [10]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Model loaded.")

Model loaded.


## Encode All Chunks

We embed every chunk's `search_text` (which includes the title and metadata).

Setting `normalize_embeddings=True` scales every vector to unit length.
This means cosine similarity equals dot product, which is faster to compute.

In [11]:
chunk_texts = chunks_df["search_text"].tolist()

chunk_embeddings = model.encode(
    chunk_texts,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Chunk embeddings shape:", chunk_embeddings.shape)
print("Rows = chunks, Columns = embedding dimensions")
print("First embedding vector norm:", np.linalg.norm(chunk_embeddings[0]))

Chunk embeddings shape: (33, 384)
Rows = chunks, Columns = embedding dimensions
First embedding vector norm: 1.0


## Encode a Query

The query must be embedded using the **same model** as the documents.

If they use different models, their vectors cannot be compared.

In [12]:
test_query = "How can I get my money back after dropping classes?"

test_query_embedding = model.encode(
    [test_query],
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Query embedding shape:", test_query_embedding.shape)

# Show top similarity scores
scores = cosine_similarity(test_query_embedding, chunk_embeddings).flatten()
top_indices = np.argsort(scores)[::-1][:5]

pd.DataFrame({
    "document_id": chunks_df.iloc[top_indices]["document_id"].values,
    "title": chunks_df.iloc[top_indices]["title"].values,
    "score": scores[top_indices]
})

Query embedding shape: (1, 384)


,document_id,title,score
0,4,Tuition Refund Policy,0.578299
1,5,Course Withdrawal Rules,0.499525
2,4,Tuition Refund Policy,0.476934
3,16,Refund Notice for Spring 2024,0.456560
4,5,Course Withdrawal Rules,0.449177


## The Semantic Aha Moment

The query says `money back` and `dropping classes`.
The relevant documents say `refund` and `withdrawal`.

The embedding model understands these are semantically equivalent.
TF-IDF and BM25 do not.

In [13]:
def retrieve_top_k_semantic(query, k=3):
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    scores = cosine_similarity(query_embedding, chunk_embeddings).flatten()
    ranking = np.argsort(scores)[::-1][:k]

    results = chunks_df.iloc[ranking].copy()
    results["score"] = scores[ranking]
    results["retriever"] = "Embeddings"
    return results[["retriever", "chunk_id", "document_id", "title",
                    "effective_date", "is_current", "score", "chunk_text"]].reset_index(drop=True)


retrieve_top_k_semantic("How can I get my money back after dropping classes?", k=5)

,retriever,chunk_id,document_id,title,effective_date,is_current,score,chunk_text
0,Embeddings,doc4_chunk0,4,Tuition Refund Policy,2025-08-15,True,0.578299,"Students who withdraw from all courses during the first 7 calendar days of the semester may receive an 80 percent tuition refund. Withdrawals during days 8 to 14 may receive a 50 percent refund. Registration fees, la..."
1,Embeddings,doc5_chunk1,5,Course Withdrawal Rules,2025-08-20,True,0.499525,the separate tuition refund policy. Students who remain enrolled in other courses may still owe full fees depending on timing and load.
2,Embeddings,doc4_chunk1,4,Tuition Refund Policy,2025-08-15,True,0.476934,"may receive a 50 percent refund. Registration fees, late-payment penalties, and technology fees are non-refundable. Refund processing normally takes 10 working days after approval."
3,Embeddings,doc16_chunk0,16,Refund Notice for Spring 2024,2024-01-10,False,0.456560,"Effective for Spring 2024 only, students who withdraw during the first 10 calendar days may receive a 75 percent tuition refund. This temporary schedule replaced the normal refund table for that term only."
4,Embeddings,doc5_chunk0,5,Course Withdrawal Rules,2025-08-20,True,0.449177,Withdrawing from a course removes the student from academic attendance in that course. Withdrawal approval does not automatically mean tuition will be refunded. Refund eligibility is governed by the separate tuition ...


# Section 10 — Hybrid Retriever

Hybrid retrieval combines lexical and semantic signals:

$$\text{hybrid score} = \alpha \times \text{semantic score} + (1 - \alpha) \times \text{lexical score}$$

Where:
- $\alpha$ = weight for semantic retrieval (0 to 1)
- $1 - \alpha$ = weight for TF-IDF lexical retrieval

Before combining, we normalize both score sets to [0, 1] range because BM25 and cosine similarity are on different scales.

In [14]:
def min_max_normalize(scores):
    scores = np.array(scores, dtype=float)
    min_score, max_score = scores.min(), scores.max()
    if max_score == min_score:
        return np.zeros_like(scores)
    return (scores - min_score) / (max_score - min_score)


def retrieve_top_k_hybrid(query, alpha=0.6, k=3):
    # TF-IDF scores
    lexical_query_vector = tfidf_vectorizer.transform([normalize_lexical_text(query)])
    lexical_scores = cosine_similarity(lexical_query_vector, tfidf_matrix).flatten()

    # Embedding scores
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )
    semantic_scores = cosine_similarity(query_embedding, chunk_embeddings).flatten()

    # Normalize and combine
    combined_scores = alpha * min_max_normalize(semantic_scores) + (1 - alpha) * min_max_normalize(lexical_scores)
    ranking = np.argsort(combined_scores)[::-1][:k]

    results = chunks_df.iloc[ranking].copy()
    results["tfidf_score"] = lexical_scores[ranking]
    results["semantic_score"] = semantic_scores[ranking]
    results["score"] = combined_scores[ranking]
    results["retriever"] = f"Hybrid alpha={alpha}"
    return results[["retriever", "chunk_id", "document_id", "title",
                    "effective_date", "is_current", "tfidf_score", "semantic_score",
                    "score", "chunk_text"]].reset_index(drop=True)


retrieve_top_k_hybrid("How can I get my money back after dropping classes?", alpha=0.6, k=5)

,retriever,chunk_id,document_id,title,effective_date,is_current,tfidf_score,semantic_score,score,chunk_text
0,Hybrid alpha=0.6,doc10_chunk0,10,Financial Aid Disbursement Notice,2025-09-03,True,0.075501,0.435509,0.841348,"Approved financial aid is usually disbursed after enrollment verification is complete. Students may see a delay if required documents are missing, if registration changes after the census date, or if the bank account..."
1,Hybrid alpha=0.6,doc4_chunk1,4,Tuition Refund Policy,2025-08-15,True,0.051903,0.476934,0.762358,"may receive a 50 percent refund. Registration fees, late-payment penalties, and technology fees are non-refundable. Refund processing normally takes 10 working days after approval."
2,Hybrid alpha=0.6,doc13_chunk0,13,Graduation Clearance Checklist,2025-09-14,True,0.072429,0.334472,0.712816,"Graduation clearance requires confirmation from the library, finance office, housing office, and academic department. Unpaid fines, overdue books, missing equipment, or unresolved tuition balances can block clearance..."
3,Hybrid alpha=0.6,doc4_chunk0,4,Tuition Refund Policy,2025-08-15,True,0.000000,0.578299,0.600000,"Students who withdraw from all courses during the first 7 calendar days of the semester may receive an 80 percent tuition refund. Withdrawals during days 8 to 14 may receive a 50 percent refund. Registration fees, la..."
4,Hybrid alpha=0.6,doc14_chunk1,14,Housing Move-Out Instructions,2025-05-30,True,0.068575,0.205709,0.549331,is billed after final inspection. Students should clear all personal items before departure.


# Section 11 — Evaluate All Three Retrievers

We compare TF-IDF, BM25, Embeddings, and Hybrid on all 13 queries using the same metrics.

In [15]:
K = 3

tfidf_eval = evaluate_retriever(
    retriever_name="TF-IDF",
    retrieval_function=lambda query, k: retrieve_top_k_tfidf(query, k),
    ground_truth=ground_truth,
    k=K
)

bm25_eval = evaluate_retriever(
    retriever_name="BM25",
    retrieval_function=lambda query, k: retrieve_top_k_bm25(query, k),
    ground_truth=ground_truth,
    k=K
)

embedding_eval = evaluate_retriever(
    retriever_name="Embeddings",
    retrieval_function=lambda query, k: retrieve_top_k_semantic(query, k),
    ground_truth=ground_truth,
    k=K
)

hybrid_eval = evaluate_retriever(
    retriever_name="Hybrid alpha=0.6",
    retrieval_function=lambda query, k: retrieve_top_k_hybrid(query, alpha=0.6, k=k),
    ground_truth=ground_truth,
    k=K
)

all_eval = pd.concat([tfidf_eval, bm25_eval, embedding_eval, hybrid_eval], ignore_index=True)

summary_df = all_eval.groupby("retriever")[
    [f"precision@{K}", f"recall@{K}", f"hit_rate@{K}", "reciprocal_rank"]
].mean().sort_values(by="reciprocal_rank", ascending=False)

summary_df

,precision@3,recall@3,hit_rate@3,reciprocal_rank
retriever,,,,
Embeddings,0.410256,0.935897,1.000000,0.961538
Hybrid alpha=0.6,0.384615,0.897436,1.000000,0.923077
BM25,0.358974,0.858974,0.923077,0.884615
TF-IDF,0.358974,0.858974,0.923077,0.884615


## Test Multiple Alpha Values for Hybrid

Alpha is a design choice. We test three values to understand the tradeoff.

In [16]:
hybrid_evals = []
for alpha in [0.2, 0.5, 0.8]:
    ev = evaluate_retriever(
        retriever_name=f"Hybrid alpha={alpha}",
        retrieval_function=lambda query, k, a=alpha: retrieve_top_k_hybrid(query, alpha=a, k=k),
        ground_truth=ground_truth,
        k=K
    )
    hybrid_evals.append(ev)

alpha_summary = pd.concat(
    [tfidf_eval, bm25_eval, embedding_eval] + hybrid_evals,
    ignore_index=True
).groupby("retriever")[
    [f"precision@{K}", f"recall@{K}", f"hit_rate@{K}", "reciprocal_rank"]
].mean().sort_values(by="reciprocal_rank", ascending=False)

alpha_summary

,precision@3,recall@3,hit_rate@3,reciprocal_rank
retriever,,,,
Embeddings,0.410256,0.935897,1.000000,0.961538
Hybrid alpha=0.8,0.384615,0.897436,1.000000,0.961538
Hybrid alpha=0.5,0.384615,0.897436,1.000000,0.910256
BM25,0.358974,0.858974,0.923077,0.884615
Hybrid alpha=0.2,0.358974,0.858974,0.923077,0.884615
TF-IDF,0.358974,0.858974,0.923077,0.884615


---

# Part 2 — Context Building

## From Retrieved Chunks to Usable Evidence

Retrieval gives us candidate evidence.

Context building decides what the model actually sees.

```text
candidate chunks
→ filter outdated
→ remove duplicates
→ sort by relevance and currency
→ apply word budget
→ label with metadata
→ context package
```

If you paste raw retrieval output into a prompt, the model may:

- use an outdated source as if it were current
- repeat the same fact from multiple chunks
- see irrelevant chunks that dilute attention
- exceed the token budget

## What Good Context Looks Like

Good context is:

| Property | Why it matters |
|---|---|
| Relevant | Only evidence related to the query |
| Current | Prefer `is_current=True` sources |
| Non-redundant | No near-duplicate chunks |
| Compact | Stays within a word budget |
| Labeled | Title, date, and current/outdated marker included |

# Section 13 — Build a Context Package

In [ ]:
def build_context_package(
    query,
    retrieval_k=8,
    alpha=0.6,
    max_context_chunks=3,
    max_chunks_per_document=1,
    word_budget=150,
    prefer_current=True,
    min_score_ratio=0.40,
    min_absolute_score=0.05
):
   
    candidates = retrieve_top_k_hybrid(query, alpha=alpha, k=retrieval_k).copy()

    if prefer_current:
        candidates = candidates.sort_values(
            by=["is_current", "score", "effective_date"],
            ascending=[False, False, False]
        ).reset_index(drop=True)

    max_score = candidates["score"].max() if len(candidates) else 0.0
    selected_rows = []
    seen_texts = set()
    per_document_counts = {}
    used_words = 0

    for _, row in candidates.iterrows():
        # Score filters
        if row["score"] < min_absolute_score:
            continue
        if max_score > 0 and row["score"] < max_score * min_score_ratio:
            continue

        # Deduplication
        normalized = re.sub(r"\s+", " ", row["chunk_text"]).strip().lower()
        if normalized in seen_texts:
            continue

        # Per-document limit
        doc_count = per_document_counts.get(row["document_id"], 0)
        if doc_count >= max_chunks_per_document:
            continue

        # Word budget
        chunk_words = len(row["chunk_text"].split())
        if selected_rows and used_words + chunk_words > word_budget:
            continue

        selected_rows.append(row.to_dict())
        seen_texts.add(normalized)
        per_document_counts[row["document_id"]] = doc_count + 1
        used_words += chunk_words

        if len(selected_rows) >= max_context_chunks:
            break

    # Build labeled context text
    blocks = []
    for position, row in enumerate(selected_rows, start=1):
        currency_label = "CURRENT" if row["is_current"] else "OUTDATED"
        blocks.append(
            f"[Source {position}] {row['title']} | {row['effective_date']} | {currency_label}\n"
            f"{row['chunk_text']}"
        )

    return {
        "query": query,
        "candidates": candidates,
        "selected_df": pd.DataFrame(selected_rows),
        "context_text": "\n\n".join(blocks),
        "used_words": used_words,
        "num_sources": len(selected_rows),
    }

# Section 14 — Inspect Context Packages

In [35]:
refund_package = build_context_package(
    query="How can I get my money back after dropping classes?",
    retrieval_k=10
)

print(f"Word budget used: {refund_package['used_words']}")
print(f"Sources selected: {refund_package['num_sources']}")
print()
print("=== Context Package ===")
print(refund_package["context_text"])

Word budget used: 99
Sources selected: 3

=== Context Package ===
[Source 1] Financial Aid Disbursement Notice | 2025-09-03 | CURRENT
Approved financial aid is usually disbursed after enrollment verification is complete. Students may see a delay if required documents are missing, if registration changes after the census date, or if the bank account details are invalid. Receiving financial

[Source 2] Tuition Refund Policy | 2025-08-15 | CURRENT
may receive a 50 percent refund. Registration fees, late-payment penalties, and technology fees are non-refundable. Refund processing normally takes 10 working days after approval.

[Source 3] Graduation Clearance Checklist | 2025-09-14 | CURRENT
Graduation clearance requires confirmation from the library, finance office, housing office, and academic department. Unpaid fines, overdue books, missing equipment, or unresolved tuition balances can block clearance. Transcript release may be delayed until all holds are removed.


In [36]:
# View the selected chunks with metadata
if len(refund_package["selected_df"]) > 0:
    refund_package["selected_df"][["title", "effective_date", "is_current", "score", "chunk_text"]]

## Current vs Outdated Conflict

For the printing query, we have a current policy (2025) and an outdated notice (2023).

The retriever may return both. Context building should prefer the current one and flag the conflict.

In [37]:
# First: see what the retriever returns (including outdated sources)
print("=== Raw retrieval for printing query ===")
retrieve_top_k_hybrid(
    query="How much is color printing now?",
    alpha=0.6,
    k=5
)[["document_id", "title", "effective_date", "is_current", "score", "chunk_text"]]

=== Raw retrieval for printing query ===


,document_id,title,effective_date,is_current,score,chunk_text
0,17,Old Printing Price Notice,2023-11-01,False,1.000000,"Before the 2025 pricing update, color printing cost 0.35 USD per page and black-and-white printing cost 0.08 USD per page after quota. This notice is retained for record purposes and should not be used for current bi..."
1,6,Campus Printing Guide,2025-09-05,True,0.886533,"Students receive a subsidized printing quota of 150 black-and-white pages each semester. Black-and-white pages after the quota cost 0.10 USD per page, while color printing costs 0.50 USD per page from the first page...."
2,6,Campus Printing Guide,2025-09-05,True,0.398963,per page from the first page. Failed print jobs may be refunded if reported to the lab supervisor within 24 hours with the print job reference number.
3,12,Student ID Card Replacement,2025-08-28,True,0.362544,the library for one day but does not work for printing or attendance scanners.
4,0,Library Borrowing Policy,2025-08-01,True,0.264120,Students may borrow up to 8 printed books at one time for 14 days. Renewals are allowed twice unless another student has placed a hold. Graduating students must clear all outstanding loans and fines before clearance ...


In [38]:
# After context building: outdated sources should be filtered or ranked last
printing_package = build_context_package(
    query="How much is color printing now?",
    retrieval_k=8,
    prefer_current=True
)

print("=== Context Package — Printing Query ===")
print(printing_package["context_text"])

=== Context Package — Printing Query ===
[Source 1] Campus Printing Guide | 2025-09-05 | CURRENT
Students receive a subsidized printing quota of 150 black-and-white pages each semester. Black-and-white pages after the quota cost 0.10 USD per page, while color printing costs 0.50 USD per page from the first page. Failed print jobs may

[Source 2] Old Printing Price Notice | 2023-11-01 | OUTDATED
Before the 2025 pricing update, color printing cost 0.35 USD per page and black-and-white printing cost 0.08 USD per page after quota. This notice is retained for record purposes and should not be used for current billing questions.


# Section 15 — Context Failure Analysis

Not all failures happen at retrieval.

A context failure happens when the right chunks were retrieved but the wrong ones were kept.

Classify failures into:

| Layer | What it means |
|---|---|
| Retrieval failure | The correct document was never ranked in the top-K |
| Context failure | Correct chunk was retrieved but filtered or displaced |
| Prompt failure | Context was good but the prompt caused the model to ignore it |
| Generation failure | Prompt was correct but model still answered incorrectly |

---

# Part 3 — Prompt Writing for LLMs in RAG

## How to Write Prompts That Use Retrieved Evidence Well

The prompt controls how the model uses the context:

- whether it stays grounded in the sources
- whether it cites where the answer came from
- whether it refuses when evidence is missing
- how it handles outdated or conflicting sources
- whether it answers briefly or structurally

A good context package plus a weak prompt can still produce a bad answer.
A strong prompt cannot rescue weak retrieval or weak context.

## Prompt Anatomy

The core parts of a strong RAG prompt:

| Part | Purpose |
|---|---|
| Role | Tell the model who it is |
| Task | Tell the model what to do |
| Evidence boundary | Tell the model what it may use |
| Decision rules | Handle missing evidence, conflicts, uncertainty |
| Output format | Control length, structure, citation style |
| Refusal behavior | What to say when the context is not enough |

# Section 16 — Build Example Context for Prompt Testing

In [26]:
sample_context = refund_package["context_text"]
sample_query = "How can I get my money back after dropping classes?"

print("=== Sample Context ===")
print(sample_context)

=== Sample Context ===
[Source 1] Financial Aid Disbursement Notice | 2025-09-03 | CURRENT
Approved financial aid is usually disbursed after enrollment verification is complete. Students may see a delay if required documents are missing, if registration changes after the census date, or if the bank account details are invalid. Receiving financial

[Source 2] Tuition Refund Policy | 2025-08-15 | CURRENT
may receive a 50 percent refund. Registration fees, late-payment penalties, and technology fees are non-refundable. Refund processing normally takes 10 working days after approval.

[Source 3] Graduation Clearance Checklist | 2025-09-14 | CURRENT
Graduation clearance requires confirmation from the library, finance office, housing office, and academic department. Unpaid fines, overdue books, missing equipment, or unresolved tuition balances can block clearance. Transcript release may be delayed until all holds are removed.


# Section 17 — Three Prompt Styles

Each prompt style is a tradeoff, not a fixed best answer.

The right style depends on:
- how sensitive the answer is
- how much control you need over grounding
- whether the user expects a formal or conversational response

In [ ]:
def build_weak_prompt(query, context_text):
    return f"""Answer the question using the context.

Question:
{query}

Context:
{context_text}
"""


def build_better_prompt(query, context_text):
    return f"""You are a careful university support assistant.

Answer using only the provided context.

Rules:
1. Do not use outside knowledge.
2. If the context is not enough to answer, say so clearly.
3. If sources disagree, prefer the most current source and mention the conflict.
4. Cite the source numbers you use in your answer.
5. Keep the answer concise but complete.

Question:
{query}

Context:
{context_text}
"""


def build_strict_prompt(query, context_text):
    return f"""You are a grounded RAG assistant.

Rules:
1. Use only the provided context. Never add background knowledge.
2. If the answer is not in the context, say: "The provided sources do not contain enough information to answer this question."
3. If a source is marked OUTDATED, do not use it as the primary answer. Mention it only to note the conflict.
4. If current and outdated sources conflict, state the conflict and use the CURRENT source.
5. Output exactly two sections:
   Answer: [your grounded answer]
   Sources: [list the source numbers you used]

Question:
{query}

Context:
{context_text}
"""


# Show all three prompts
prompt_table = pd.DataFrame([
    {"style": "weak",   "prompt": build_weak_prompt(sample_query, sample_context)},
    {"style": "better", "prompt": build_better_prompt(sample_query, sample_context)},
    {"style": "strict", "prompt": build_strict_prompt(sample_query, sample_context)},
])

prompt_table

,style,prompt
0,weak,Answer the question using the context.\n\nQuestion:\nHow can I get my money back after dropping classes?\n\nContext:\n[Source 1] Financial Aid Disbursement Notice | 2025-09-03 | CURRENT\nApproved financial aid is usu...
1,better,"You are a careful university support assistant.\n\nAnswer using only the provided context.\n\nRules:\n1. Do not use outside knowledge.\n2. If the context is not enough to answer, say so clearly.\n3. If sources disagr..."
2,strict,"You are a grounded RAG assistant.\n\nRules:\n1. Use only the provided context. Never add background knowledge.\n2. If the answer is not in the context, say: ""The provided sources do not contain enough information to ..."


In [28]:
# Print each prompt clearly
for _, row in prompt_table.iterrows():
    print(f"{'='*60}")
    print(f"STYLE: {row['style'].upper()}")
    print(f"{'='*60}")
    print(row["prompt"])
    print()

STYLE: WEAK
Answer the question using the context.

Question:
How can I get my money back after dropping classes?

Context:
[Source 1] Financial Aid Disbursement Notice | 2025-09-03 | CURRENT
Approved financial aid is usually disbursed after enrollment verification is complete. Students may see a delay if required documents are missing, if registration changes after the census date, or if the bank account details are invalid. Receiving financial

[Source 2] Tuition Refund Policy | 2025-08-15 | CURRENT
may receive a 50 percent refund. Registration fees, late-payment penalties, and technology fees are non-refundable. Refund processing normally takes 10 working days after approval.

[Source 3] Graduation Clearance Checklist | 2025-09-14 | CURRENT
Graduation clearance requires confirmation from the library, finance office, housing office, and academic department. Unpaid fines, overdue books, missing equipment, or unresolved tuition balances can block clearance. Transcript release may be de

# Section 18 — When Each Prompt Style Fits

In [29]:
comparison = pd.DataFrame([
    {
        "style": "weak",
        "best for": "quick experiments, low-stakes questions",
        "risk": "hallucination, no evidence boundary, no citations",
        "bad for": "any task where grounding matters"
    },
    {
        "style": "better",
        "best for": "most RAG use cases — balanced grounding and flexibility",
        "risk": "still soft enough for the model to drift slightly",
        "bad for": "strict schema requirements or sensitive factual tasks"
    },
    {
        "style": "strict",
        "best for": "factual, sensitive queries where sources must be cited exactly",
        "risk": "too rigid for open-ended or nuanced questions",
        "bad for": "questions that need explanation or creative framing"
    },
])

comparison

,style,best for,risk,bad for
0,weak,"quick experiments, low-stakes questions","hallucination, no evidence boundary, no citations",any task where grounding matters
1,better,most RAG use cases — balanced grounding and flexibility,still soft enough for the model to drift slightly,strict schema requirements or sensitive factual tasks
2,strict,"factual, sensitive queries where sources must be cited exactly",too rigid for open-ended or nuanced questions,questions that need explanation or creative framing


# Section 19 — Prompt Failure Cases

A prompt can fail even with good context and good retrieval.

| Failure | Cause | Fix |
|---|---|---|
| Hallucination | No evidence boundary rule | Add: "Use only the context" |
| Evidence drift | No citation requirement | Add: "Cite source numbers" |
| Outdated answer | No currency check | Add: "Prefer CURRENT sources" |
| Over-confident | No uncertainty rule | Add: "Say so if the context is insufficient" |
| Vague output | No format instruction | Add: "Answer in two sections: Answer / Sources" |

In [30]:
# Build a context with a current/outdated conflict for the printing query
printing_conflict_context = printing_package["context_text"]
printing_query = "How much is color printing now?"

print("=== Weak Prompt for Printing Query ===")
print(build_weak_prompt(printing_query, printing_conflict_context))

print()
print("=== Strict Prompt for Printing Query ===")
print(build_strict_prompt(printing_query, printing_conflict_context))

=== Weak Prompt for Printing Query ===
Answer the question using the context.

Question:
How much is color printing now?

Context:
[Source 1] Campus Printing Guide | 2025-09-05 | CURRENT
Students receive a subsidized printing quota of 150 black-and-white pages each semester. Black-and-white pages after the quota cost 0.10 USD per page, while color printing costs 0.50 USD per page from the first page. Failed print jobs may

[Source 2] Old Printing Price Notice | 2023-11-01 | OUTDATED
Before the 2025 pricing update, color printing cost 0.35 USD per page and black-and-white printing cost 0.08 USD per page after quota. This notice is retained for record purposes and should not be used for current billing questions.


=== Strict Prompt for Printing Query ===
You are a grounded RAG assistant.

Rules:
1. Use only the provided context. Never add background knowledge.
2. If the answer is not in the context, say: "The provided sources do not contain enough information to answer this question."
3

## The Connection Between Parts 1, 2, and 3

```text
Bad retrieval → wrong chunks → wrong context → no prompt can fix it
Good retrieval + good context → weak prompt → model ignores evidence
Good retrieval + good context + strong prompt → grounded answer
```

Each part depends on the part before it.

A problem at any layer propagates forward.

---

# Final Assignment

You will build a complete RAG pipeline on your own knowledge base.

## Task 1 — Build Your Own Corpus

Create at least 15 documents from one domain.

Allowed examples: hospital FAQ, hotel policies, software documentation, library help desk, transportation information.

Requirements:
- Each document must have: `document_id`, `title`, `doc_type`, `effective_date`, `is_current`, `text`
- Include at least 2 outdated documents that conflict with current ones
- Include at least 3 paraphrase traps (query wording differs from document wording)

## Task 2 — Create Queries and Ground Truth

Create at least 10 queries with manually defined ground truth.

At least 4 queries must use different wording from the relevant document.

Example:
```python
ground_truth = {
    "How do I get my money back?": [0],      # paraphrase: document says 'refund'
    "Can I walk in without an appointment?": [7],  # paraphrase: document says 'walk-ins'
}
```

## Task 3 — Build and Evaluate Three Retrievers

Implement:
1. TF-IDF retriever
2. Embedding retriever using `SentenceTransformer("all-MiniLM-L6-v2")`
3. Hybrid retriever combining both

Evaluate all three using Precision@3, Recall@3, Hit Rate@3, and MRR.

Test at least three alpha values for hybrid retrieval and report which worked best.

## Task 4 — Build a Context Package

Build a context package for at least 5 queries that:

- filters outdated sources
- removes near-duplicate chunks
- respects a word budget of 150 words
- labels each source with title, date, and CURRENT/OUTDATED

Show at least one current/outdated conflict and explain how you handled it.

## Task 5 — Write Three Prompts

For the same query and context package:

1. Write a weak prompt and explain why it is weak
2. Write a better prompt with grounding rules and citation requirement
3. Write a strict prompt with a structured two-part output

For each style, describe one scenario where that style is the wrong choice.

## Task 6 — Error Analysis

Choose at least 3 failed queries from your evaluation.

For each one, answer:

1. The query
2. The correct document
3. What the retriever returned instead
4. Which layer failed: retrieval, context, prompt, or generation?
5. How would you fix it?

---

# Final Takeaways

**Retrieval**
- TF-IDF and BM25 are lexical — strong on exact words, weak on synonyms
- Embeddings are semantic — strong on meaning, weak on exact codes and numbers
- Hybrid is often the best practical baseline because it combines both signals
- No retriever is universally best — measure on your data

**Context Building**
- Retrieved chunks are candidate evidence, not final context
- Outdated sources must be filtered or flagged
- Deduplication, ordering, and word budgets all matter
- Better context usually produces better grounded answers

**Prompt Writing**
- Prompts are control instructions, not just text wrappers
- Weak prompts invite hallucination even with good context
- Strict prompts force grounding but can be too rigid for nuanced questions
- The better prompt is the right default for most RAG teaching cases

**The Pipeline**
```text
retrieval quality → context quality → answer quality
```
A failure at any layer propagates forward. Fix the layer, not the symptom.